## Assetwise sign-in

In order to use chrome from your scripting, you should download the relevant [chrome driver](https://googlechromelabs.github.io/chrome-for-testing/#stable) (chromedriver, not chrome!) for your current version of google chrome, then add the path to the driver to your script below.

In [1]:
# User Configuration
user_email = "dane.parks@dot.ohio.gov"
path_to_excel = r"C:\Users\dparks1\Desktop\assetwise_script_demo\reportSummaryData.xlsx"
file_save_path = r"C:\Users\dparks1\Desktop\assetwise_script_demo"

In [2]:
import os
import time
import shutil
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.select import Select
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [4]:
# Launch the development/automation browser
# options = webdriver.ChromeOptions()
# options.add_experimental_option(
#     "prefs",
#     {
#         "download.default_directory": file_save_path,  # Change default directory for downloads
#         "download.prompt_for_download": False,  # To auto download the file
#         "download.directory_upgrade": True,
#         "plugins.always_open_pdf_externally": True,  # It will not show PDF directly in chrome
#     },
# )

driver = webdriver.Edge()

In [ ]:
driver.maximize_window()
driver.get("https://ohiodot-it.bentley.com/login.aspx")
print(f"Page title: {driver.title}")

# Wait until the username box appears, then select it
WebDriverWait(driver, timeout=5).until(
    lambda d: d.find_element(By.XPATH, '//*[@id="ContentPlaceHolder1_cmdOIDCLogin"]')
)

IMS_Login = driver.find_element(By.XPATH, '//*[@id="ContentPlaceHolder1_cmdOIDCLogin"]')
IMS_Login.click()

In [ ]:
# Sign in
username_input = driver.find_element(By.XPATH, '//*[@id="identifierInput"]').send_keys(
    user_email
)
submit_button = driver.find_element(By.XPATH, '//*[@id="sign-in-button"]')
submit_button.click()

In [5]:
# Get the list of SFNs from the excel sheet
import pandas as pd

# Load the excel sheet, 5th tab, Column 'R'=index 17
df = pd.read_excel(path_to_excel, sheet_name=0)

In [6]:
# Store Column in list
sfn_list = df["Asset Code"].tolist()

## Loop through the list of SFNs

In [7]:
def find_first_iframe(driver):
    # Find all iframes on the current page
    iframes = driver.find_elements(By.TAG_NAME, "iframe")

    driver.switch_to.frame(iframes[1])  # Switch to the iframe by index

    try:
        # Try to locate and click the element inside the iframe
        view_pdf_1 = driver.find_element(
            By.XPATH,
            '//*[@id="ctl00_ContentPlaceHolder1_upRASContent"]/table/tbody/tr[3]/td[2]/span',
        )
        view_pdf_1.click()
        print("Element found and clicked!")
        return  # Exit the function after successful interaction
    except Exception as e:
        # Handle case where element is not found in this iframe
        print(f"Element not found in iframe: {e}")
        pass

    # Switch back to the parent frame after checking the current iframe
    driver.switch_to.parent_frame()

In [8]:
def find_second_iframe(driver):
    # Find all iframes on the current page
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    
    driver.switch_to.frame(iframes[1])  # Switch to the iframe by index
    print(f"Switched to iframe")

    try:
        # Wait for the element inside the iframe
        view_pdf_2 = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="ContentPlaceHolder1_printSections_cmdViewPDFBottom"]')
            )
        )
        view_pdf_2.click()
        print("Element found and clicked in second iframe!")
        return  # Exit after successful interaction
    except Exception as e:
        # //TODO - Figure out which frames these are in and remove the errors
        # print(f"Element not found in iframe {index}: {e}")
        pass

    # Switch back to the parent frame after checking the current iframe
    driver.switch_to.parent_frame()

In [9]:
# Temporarily store an SFN in a variable for testing
for sfn in sfn_list:
    # Hacky, but the loading screen breaks the script...
    time.sleep(10)
    
    # Select the search bar
    search_bar = input_element = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, "//input[@placeholder='Search Assets']"))
    )
    search_bar.click()

    search_bar.send_keys(sfn)
    search_bar.send_keys(Keys.RETURN)

    try:
        # Wait until the iframe is present and locate it
        iframe = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "iframe"))  # Locate the iframe by tag name
        )
    
        # Switch to the iframe context
        driver.switch_to.frame(iframe)
    
        # Now, locate the link <a> inside the iframe
        link_element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//a[contains(@href, 'bridgedetail.aspx')]"))
        )
    
        # Click the link or extract the href
        url = link_element.get_attribute("href")
        print(f"URL found: {url}")
        driver.get(url)
    
        # Optionally: Switch back to the main content
        driver.switch_to.default_content()
    
    except Exception as e:
        print(f"An error occurred: {e}")

    wait = WebDriverWait(driver, timeout=20)
    wait_1 = wait.until(
            EC.element_to_be_clickable((By.XPATH, '//*[@id="tabAsset"]/span/span/span'))
        )
    asset_info = driver.find_element(By.XPATH, '//*[@id="tabAsset"]/span/span/span')
    asset_info.click()
    
    wait_2 = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                '//*[@id="completedReportsGridDiv"]/div/div[2]/div/div[1]/div[3]/div[1]/div/div[10]/div',
            )
        )
    )
    hamburger = driver.find_element(
        By.XPATH,
        '//*[@id="completedReportsGridDiv"]/div/div[2]/div/div[1]/div[3]/div[1]/div/div[10]/div',
    )
    hamburger.click()
    
    time.sleep(1)
    
    # finds the button in the first iframe
    find_first_iframe(driver)
    
    driver.switch_to.parent_frame()
    
    # finds the button in the second iframe
    find_second_iframe(driver)

    driver.switch_to.parent_frame()
    
    close_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.CLASS_NAME, "rwCloseButton"))
    )
    
    close_button.click()

URL found: https://ohiodot-it.bentley.com/bridgedetail.aspx?type=0&as_id=79723
Element found and clicked!
Switched to iframe
Element found and clicked in second iframe!
URL found: https://ohiodot-it.bentley.com/bridgedetail.aspx?type=0&as_id=79724
Element found and clicked!
Switched to iframe
Element found and clicked in second iframe!
URL found: https://ohiodot-it.bentley.com/bridgedetail.aspx?type=0&as_id=79725
Element found and clicked!
Switched to iframe
Element found and clicked in second iframe!
URL found: https://ohiodot-it.bentley.com/bridgedetail.aspx?type=0&as_id=79726
Element found and clicked!
Switched to iframe
Element found and clicked in second iframe!
URL found: https://ohiodot-it.bentley.com/bridgedetail.aspx?type=0&as_id=79727
Element found and clicked!
Switched to iframe
Element found and clicked in second iframe!
URL found: https://ohiodot-it.bentley.com/bridgedetail.aspx?type=0&as_id=79728
Element found and clicked!
Switched to iframe
Element found and clicked in s

In [10]:
def rename_files():
    for index, sfn in enumerate(sfn_list):
        if index == 0:
            # Renames the file
            try:
                shutil.copy(
                    Path(file_save_path) / "report.pdf",
                    Path(file_save_path) / f"{sfn}_inspection_report.pdf",
                )
                os.remove(Path(file_save_path) / "report.pdf")
                time.sleep(5)
            # If source and destination are same
            except shutil.SameFileError:
                print("Source and destination represents the same file.")

            # If there is any permission issue
            except PermissionError:
                print("Permission denied.")

            # For other errors
            except:
                print("Error occurred while copying file.")
        else:
            # Renames the file
            try:
                shutil.copy(
                    Path(file_save_path) / f"report ({index}).pdf",
                    Path(file_save_path) / f"{sfn}_inspection_report.pdf",
                )
                os.remove(Path(file_save_path) / f"report ({index}).pdf")
                time.sleep(5)
            # If source and destination are same
            except shutil.SameFileError:
                print("Source and destination represents the same file.")

            # If there is any permission issue
            except PermissionError:
                print("Permission denied.")

            # For other errors
            except Exception as e:
                print(f"Error occurred while copying file. {e}")

In [11]:
rename_files()

Error occurred while copying file. [Errno 2] No such file or directory: 'C:\\Users\\dparks1\\Desktop\\assetwise_script_demo\\report (63).pdf'


# Old Cells

# Assetwise API?

[ODOT Assetwise](https://ohiodot-it.bentley.com)

API Endpoints - `https://ohiodot-it-api.bentley.com/swagger/index.html`

# //TODO - I Can Only Get the First Page of Results with This

In [5]:
import json

In [6]:
with open(r"C:\Users\dparks1\PycharmProjects\civilpy\secrets.json", 'r') as file:
    secrets = json.load(file)

In [139]:
# print(secrets['BENTLEY_ASSETWISE_API'])
# print(secrets['BENTLEY_ASSETWISE_KEY_NAME'])

In [140]:
import requests
from pydantic import BaseModel, ValidationError
from typing import List, Optional

In [141]:
# Define the Pydantic model for validating the expected response
class AssetData(BaseModel):
    ID: Optional[str]
    Name: Optional[str]
    Latitude: Optional[float]
    Longitude: Optional[float]
    Status: Optional[str]
    AssetType: Optional[str]
    ParentID: Optional[str]

In [142]:
class ApiResponse(BaseModel):
    data: List[AssetData]

In [143]:
# Function to fetch and validate data
def fetch_and_validate_data(api_endpoint: str):
    try:
        # Make the GET request
        response = requests.get(api_endpoint)

        # Raise an HTTPError for bad responses (4xx and 5xx)
        response.raise_for_status()

        # Parse the JSON response
        json_data = response.json()

        # Validate JSON response with Pydantic
        validated_data = ApiResponse(data=json_data)

        return validated_data

    except ValidationError as ve:
        print("Validation error:", ve)
    except requests.exceptions.RequestException as re:
        print("Request error:", re)

In [144]:
import requests
import base64

In [145]:
def fetch_all_pages_with_skip_top(api_endpoint, username, password):
    """
    Fetch all paginated data using `$top` and `$skip` query parameters based on the
    API documentation for pagination.
    
    Args:
        api_endpoint (str): Base API URL.
        username (str): Username for authentication.
        password (str): Password for authentication.

    Returns:
        list: A list of all items retrieved from the API.

    Raises:
        Exception: If there's an error during the request.
    """
    # Encode the username and password in 'username:password' format
    credentials = f"{username}:{password}"
    encoded_credentials = base64.b64encode(credentials.encode('utf-8')).decode('utf-8')

    # Set the Basic Authorization header
    headers = {
        "Authorization": f"Basic {encoded_credentials}"
    }

    # Initialize variables for pagination
    all_data = []
    skip = 0  # Start with 0 items skipped
    top = 100  # Default page size (adjust if needed and within limits)
    page_count = 1  # Keep track of the current page

    while page_count < 5:
        # Add $top and $skip parameters
        params = {
            "$top": top,
            "$skip": skip
        }

        print(f"Fetching page {page_count} with $skip={skip} and $top={top}...")

        # Send the GET request
        response = requests.get(api_endpoint, headers=headers, params=params)

        # Raise error if the request failed
        response.raise_for_status()

        # Parse the JSON response
        json_data = response.json()

        # Fetch the data from the response
        page_data = json_data.get('data', [])
        all_data.extend(page_data)

        # Debugging information
        print(f"Fetched {len(page_data)} items from page {page_count}.")

        # Break the loop if fewer records are returned than `top` (last page)
        if len(page_data) < top:
            print("Reached the last page. No more data to fetch.")
            break

        # Increment the `skip` value for the next page
        skip += top
        page_count += 1

    return all_data

In [146]:
# Example usage
if __name__ == "__main__":
    # API base URL
    api_url = "https://ohiodot-it-api.bentley.com/api/Asset?IncludeCoordinates=true&IncludeParent=false"
    username = secrets['BENTLEY_ASSETWISE_KEY_NAME']  # Replace with your username
    password = secrets['BENTLEY_ASSETWISE_API']  # Replace with your password or API key

    try:
        all_data = fetch_all_pages_with_skip_top(api_url, username, password)
        print(f"\nTotal items fetched: {len(all_data)}")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

Fetching page 1 with $skip=0 and $top=100...
Fetched 100 items from page 1.
Fetching page 2 with $skip=100 and $top=100...
Fetched 100 items from page 2.
Fetching page 3 with $skip=200 and $top=100...
Fetched 100 items from page 3.
Fetching page 4 with $skip=300 and $top=100...
Fetched 100 items from page 4.

Total items fetched: 400


In [137]:
for value in all_data:
    if value['as_id'] == 282540:
        print('1')

1
1
1
1


# Example of Extracting Data and Rendering a Model with It

In [ ]:
import cadquery as cq

In [147]:
class CADObject:
    def __init__(self, length=1.0, width=1.0, height=1.0):
        self.length = length
        self.width = width
        self.height = height

In [ ]:
def generate_cad_model(cad_object):
    """
    Generate a 3D model (e.g., a cube) based on the CADObject's dimensions.
    """
    model = cq.Workplane("XY").box(
        cad_object.length, cad_object.width, cad_object.height
    )
    return model


In [149]:
def export_stl(model, filename="model.stl"):
    """
    Export the CAD model to an STL file for rendering.
    """
    cq.exporters.export(model, filename)

In [152]:
from nicegui import ui
import os
from cadquery import exporters
# from trimesh.exchange.stl import load_mesh  # Optional: For rendering with three.js

# Define the CAD object
cad_object = CADObject()

# Function to update the 3D model when sliders change
def update_model():
    model = generate_cad_model(cad_object)
    export_stl(model, "assets/model.stl")  # Save STL file dynamically
    scene.update()  # Refresh the 3D scene

# NiceGUI app
def main():
    ui.label("Adjust 3D Model:").classes("text-lg")

    with ui.row():
        ui.slider(min=1.0, max=10.0, step=0.1, value=cad_object.length, label="Length", on_change=lambda e: setattr(cad_object, "length", e.value) or update_model())
        ui.slider(min=1.0, max=10.0, step=0.1, value=cad_object.width, label="Width", on_change=lambda e: setattr(cad_object, "width", e.value) or update_model())
        ui.slider(min=1.0, max=10.0, step=0.1, value=cad_object.height, label="Height", on_change=lambda e: setattr(cad_object, "height", e.value) or update_model())

    # Render 3D Scene Using NiceGUI's `scene` component
    with ui.card():
        scene = ui.scene().classes("w-full h-96")
        scene.mesh('assets/model.stl')  # STL loading support in NiceGUI

ui.run(title="Dynamic 3D CAD Viewer")

disabling auto-reloading because is is only supported when running from a file


RuntimeError: asyncio.run() cannot be called from a running event loop